In [1]:
import pandas as pd
import numpy as np


df = pd.read_csv("/kaggle/input/datasets/mfaizanasghar/quranandsunnah/final_islamic_dataset.csv")

In [2]:
!pip install -q transformers datasets accelerate

In [3]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)


texts = df["english_text"].dropna().astype(str).tolist()
raw_dataset = Dataset.from_dict({"text": texts})

split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)

model_name = "distilgpt2"  # You can also use "distilgpt2" or "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)


tokenizer.pad_token = tokenizer.eos_token


# 3. Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )


tokenized_datasets = split_dataset.map(
    tokenize_function, batched=True, remove_columns=["text"]
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [4]:
# 1. Load in float16 instead of float32! This halves the memory footprint.
model = AutoModelForCausalLM.from_pretrained(
    model_name
)
model.resize_token_embeddings(len(tokenizer))

training_args = TrainingArguments(
    output_dir="./hf_islamic_gpt2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,   # Reduced from 8
    per_device_eval_batch_size=8,    # Reduced from 8
    gradient_accumulation_steps=4,   # Increased from 4 to 8 to keep effective batch size the same
    num_train_epochs=3,
    weight_decay=0.01,
    fp16= True,  
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

trainer.train()

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name
)
model.resize_token_embeddings(len(tokenizer))


In [5]:
!ls -lh ./hf_islamic_gpt2

In [ ]:
from transformers import AutoModelForCausalLM

# The Trainer saved the checkpoints to this directory. 
# Since it took 885 steps to finish 3 epochs, the final checkpoint is saved there.
checkpoint_path = "./hf_islamic_gpt2/checkpoint-885"

# Load the fine-tuned model directly from disk
model = AutoModelForCausalLM.from_pretrained(checkpoint_path)

print("Fine-tuned model successfully loaded from checkpoint!")

In [6]:
import math

# 1. Run evaluation on the test split
eval_results = trainer.evaluate()

# 2. Calculate perplexity
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print("--- Evaluation Results ---")
print(f"Validation Loss: {eval_loss:.4f}")
print(f"Perplexity:      {perplexity:.2f}")

In [7]:
from transformers import pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

In [8]:
# Compare Greedy vs Sampled Output on a single prompt
prompt = "The Messenger of Allah (ﷺ) said:"

print("--- Greedy Decoding ---")
print(
    generator(prompt, max_new_tokens=40, do_sample=False)[0]["generated_text"]
)

print("\n--- Temperature Sampling (T=0.7) ---")
print(
    generator(
        prompt,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,
    )[0]["generated_text"]
)

In [ ]:
test_prompts = [
    "The Prophet said,",
    "Indeed, Allah commands justice and",
    "Whoever believes in Allah and the Last Day should",
    "It was narrated that Abu Hurairah said:",
]

print("=" * 60)
for prompt in test_prompts:
    output = generator(
        prompt,
        max_new_tokens=60,
        num_return_sequences=1,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
    )
    print(f"Prompt: {prompt}")
    print(f"Generated:\n{output[0]['generated_text']}")
    print("-" * 60)

In [9]:
!pip install -q evaluate

In [10]:
import evaluate
from tqdm.auto import tqdm

# 1. Load the BLEU metric
bleu_metric = evaluate.load("bleu")

# 2. Select a subset of your test data to evaluate (e.g., 50 examples)
# Using the raw test split from cell 7
test_subset = split_dataset["test"]["text"][:50]

predictions = []
references_bleu = []

print("Generating continuations for BLEU evaluation...")
for text in tqdm(test_subset):
    # Split text into a prompt (first 10 words) and the expected continuation
    words = text.split()
    if len(words) < 15:
        continue  # Skip texts that are too short to split meaningfully
        
    prompt = " ".join(words[:10])
    ground_truth_continuation = " ".join(words[10:])
    
    # Generate the continuation
    # We use greedy decoding (do_sample=False) because randomness hurts BLEU scores
    output = generator(
        prompt, 
        max_new_tokens=len(words) - 10, # Try to match the original text length
        do_sample=False, 
        pad_token_id=tokenizer.eos_token_id
    )
    
    # The pipeline returns the full string (prompt + generated text). 
    # We slice out the prompt to isolate the newly generated continuation.
    full_generated_text = output[0]["generated_text"]
    generated_continuation = full_generated_text[len(prompt):].strip()
    
    predictions.append(generated_continuation)
    # BLEU expects a list of valid references for each prediction
    references_bleu.append([ground_truth_continuation])

# 3. Compute and display the results
results = bleu_metric.compute(predictions=predictions, references=references_bleu)

print("\n--- BLEU Evaluation Results ---")
print(f"Overall BLEU Score: {results['bleu']:.4f}")
print(f"N-gram Precisions (1-gram to 4-gram):")
for i, prec in enumerate(results['precisions'], 1):
    print(f"  {i}-gram: {prec:.2f}%")

In [11]:
!pip install -q sentence-transformers faiss-gpu


In [12]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


print("Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Extract texts and metadata from your existing dataframe
# Dropping any NaN values to prevent embedding errors
valid_data = df.dropna(subset=['english_text', 'source_info'])
documents = valid_data['english_text'].astype(str).tolist()
sources = valid_data['source_info'].astype(str).tolist()

# 3. Encode the documents into vectors (This takes a minute on the T4 GPU)
print(f"Encoding {len(documents)} documents into vectors...")
embeddings = embedding_model.encode(documents, show_progress_bar=True)

# 4. Build the FAISS Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
print(f"FAISS index built successfully with {index.ntotal} documents!")

In [13]:
def ask_islamic_chatbot(query, top_k=3):
    # 1. Embed the user's question
    query_vector = embedding_model.encode([query])
    
    # 2. Retrieve the top_k most relevant documents from FAISS
    distances, indices = index.search(query_vector, top_k)
    
    retrieved_texts = []
    retrieved_sources = []
    
    for i in indices[0]:
        retrieved_texts.append(documents[i])
        retrieved_sources.append(sources[i])
        
    # 3. Inject the retrieved context into a prompt for your fine-tuned model
    context = "\n".join(retrieved_texts)
    prompt = (
        f"Based on the following Islamic texts:\n{context}\n\n"
        f"Answer this question: {query}\n\n"
        f"Answer:"
    )
    
    # 4. Generate the final answer
    output = generator(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Extract only the generated text (ignoring the prompt)
    full_generated_text = output[0]['generated_text']
    answer = full_generated_text[len(prompt):].strip()
    
    # 5. Display the results nicely
    print("=" * 60)
    print(f"User Query: {query}")
    print("-" * 60)
    print(f"Chatbot Answer:\n{answer}")
    print("-" * 60)
    print("Sources Cited:")
    for source in set(retrieved_sources): # Use set() to remove duplicates
        print(f"- {source}")
    print("=" * 60)

In [14]:
ask_islamic_chatbot("What are the rights of a neighbor?")
ask_islamic_chatbot("What does the Quran say about patience?")

In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [17]:
def ask_islamic_chatbot(query, top_k=3):
    # 1. Embed the user's question
    query_vector = embedding_model.encode([query])
    
    # 2. Retrieve the top_k most relevant documents from FAISS
    distances, indices = index.search(query_vector, top_k)
    
    retrieved_texts = []
    retrieved_sources = []
    
    for i in indices[0]:
        retrieved_texts.append(documents[i])
        retrieved_sources.append(sources[i])
        
    context = "\n---\n".join(retrieved_texts)
    
    # 3. Format the prompt using Qwen's expected Chat format
    messages = [
        {
            "role": "system",
            "content": "You are a factual Islamic assistant. Use ONLY the provided context to answer the question. If the answer is not in the context, say 'I don't know based on the provided text.'"
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}"
        }
    ]
    
    # Apply the Qwen chat template
    text_prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # 4. Tokenize and move to GPU
    inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)
    
    # 5. Generate the final answer
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False, # Use False for RAG so it gives factual, grounded answers
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Slice the output to only get the newly generated text (ignoring the prompt)
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    answer = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    # 6. Display the results nicely
    print("=" * 60)
    print(f"User Query: {query}")
    print("-" * 60)
    print(f"Chatbot Answer:\n{answer}")
    print("-" * 60)
    print("Sources Cited:")
    for source in set(retrieved_sources):
        print(f"- {source}")
    print("=" * 60)

In [18]:
ask_islamic_chatbot("What are the rights of a neighbor?")
ask_islamic_chatbot("What does the Quran say about patience?")

In [19]:
!zip -r my_model_checkpoints.zip ./hf_islamic_gpt2

In [21]:
from huggingface_hub import login

# 1. Authenticate your Kaggle session
# Replace 'YOUR_WRITE_TOKEN' with your actual Hugging Face token
login(token="YOUR_HUGGINGFACE_TOKEN")

# 2. Push the model and tokenizer to your repository
model.push_to_hub("faizzanasghar/islamicgpt")
tokenizer.push_to_hub("faizzanasghar/islamicgpt")

print("Model safely pushed to Hugging Face!")

In [1]:
ask_islamic_chatbot("Who is best pm of pakistan?")

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "faizzanasghar/islamicgpt"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)